# Modeling — 5-Wochen-Vorhersage (v2)

Trainiert LightGBM auf den vom Preprocessing erzeugten Parquet-Dateien
und schreibt die Kaggle-Submission (`pred_week1..5`).

**Input** — `outputs/processed/`
- `train_labeled_v2.parquet` — wöchentlich gelabelte Zeilen inkl. Features
- `test_features_v2.parquet` — 91-Tage-Testfenster je Region inkl. Features

**Output** — `outputs/submissions/submission_<mode>_v2.csv`

**Pipeline:** Sliding-Window (Woche *i* → Score Woche *i+1..i+5*) ·
Region-Holdout · Baselines · 5 LightGBM-Modelle · Persistence-Blend.

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import lightgbm as lgb

# --- Umgebung erkennen ---
IS_KAGGLE = os.environ.get("KAGGLE_KERNEL_RUN_TYPE", "") != ""

if IS_KAGGLE:
    OUTPUTS_DIR = Path("/kaggle/working/outputs")
else:
    root = next(
        (c for c in [Path.cwd(), *Path.cwd().parents] if (c / "outputs").is_dir()),
        Path.cwd(),
    )
    OUTPUTS_DIR = root / "outputs"

PROCESSED_DIR = OUTPUTS_DIR / "processed"
SUBMISSION_DIR = OUTPUTS_DIR / "submissions"
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PARQUET = PROCESSED_DIR / "train_labeled_v2.parquet"
TEST_PARQUET = PROCESSED_DIR / "test_features_v2.parquet"

# --- Konfiguration ---
RANDOM_STATE = 42
N_WEEKS = 5            # Vorhersagehorizont (pred_week1..5)
VAL_REGION_FRAC = 0.2  # Anteil Regionen fuer den Holdout
ES_ROUNDS = 50         # Early-Stopping-Geduld
BLEND_PERSIST = 0.35   # Gewicht der Persistence im finalen Blend

print(f"Umgebung: {'Kaggle' if IS_KAGGLE else 'Lokal'}")
print(f"Train: {TRAIN_PARQUET}  ({'OK' if TRAIN_PARQUET.exists() else 'FEHLT'})")
print(f"Test:  {TEST_PARQUET}  ({'OK' if TEST_PARQUET.exists() else 'FEHLT'})")

## 1 · Daten laden

In [ ]:
# --- Daten laden ---
MODE = "sample" if "sample" in TRAIN_PARQUET.name else "full"

print("Lade Parquet-Dateien...")
train_df = pd.read_parquet(TRAIN_PARQUET)
test_df = pd.read_parquet(TEST_PARQUET)

# Meta-Spalten zaehlen nicht als numerisches Modell-Feature.
# 'region_id' wurde wieder in META aufgenommen (wir nutzen stattdessen die region_stats)
META = {"region_id", "date", "year", "month", "day", "ordinal", "score", "score_persist7"}
FEATURES = [c for c in train_df.columns if c not in META and c in test_df.columns]

train_df[FEATURES] = train_df[FEATURES].astype(np.float32)
test_df[FEATURES] = test_df[FEATURES].astype(np.float32)

print(f"Modus: {MODE}")
print(f"Train: {len(train_df):,} Zeilen · {train_df['region_id'].nunique():,} Regionen")
print(f"Test:  {len(test_df):,} Zeilen · {test_df['region_id'].nunique():,} Regionen")
print(f"Features ({len(FEATURES)}): {FEATURES[:6]} …")
train_df.head(3)


## 2 · Sliding-Window-Helfer

Die gelabelten Zeilen liegen pro Region bereits **woechentlich** vor — jede
Zeile ist eine Woche. Aus den Features der Woche *i* sagen wir die Scores der
Wochen *i+1 … i+5* voraus. Validiert wird auf **ungesehenen Regionen**
(je 1 Fenster, wie die Test-Situation).

In [ ]:
def clip_scores(arr):
    """Auf den Wettbewerbs-Score-Bereich begrenzen."""
    return np.clip(arr, 0.0, 5.0)


def mae(y_true, y_pred):
    """MAE ueber alle Regionen x 5 Wochen (Kaggle-Metrik)."""
    return float(np.mean(np.abs(clip_scores(y_pred) - np.asarray(y_true))))


def week_target(y, w):
    """Labels fuer Woche w als 1D-float64-Array (LightGBM 4.x erwartet 1D)."""
    return np.ascontiguousarray(y[:, w], dtype=np.float64)


def build_samples(weekly, mode="train", val_offset_weeks=0):
    """
    Sliding-Window je Region.
    val_offset_weeks: 0 = letztes Fenster, 5 = Fenster 5 Wochen davor, etc.
    mode='train': Alle Fenster VOR dem Validierungsfenster.
    mode='val': Genau das Validierungsfenster.
    mode='all': Alle verfuegbaren Fenster.
    """
    X_parts, y_parts, p_parts, r_parts = [], [], [], []
    for region, g in weekly.groupby("region_id", sort=True):
        g = g.sort_values("ordinal")
        scores = g["score"].to_numpy(np.float32)
        n = len(g)
        
        if n <= 2 * N_WEEKS + val_offset_weeks:
            continue
            
        y = np.lib.stride_tricks.sliding_window_view(scores[1:], N_WEEKS)
        X = g.iloc[: n - N_WEEKS][FEATURES]
        persist = scores[: n - N_WEEKS]
        
        target_val_idx = -1 - val_offset_weeks
        
        if mode == "val":
            # Das Validierungsfenster
            X, y, persist = X.iloc[[target_val_idx]], y[[target_val_idx]], persist[[target_val_idx]]
        elif mode == "train":
            # Alle Fenster davor (ohne Target-Ueberschneidung)
            # Das Fenster bei target_val_idx ueberschneidet sich die naechsten N_WEEKS.
            # Train darf hoechstens bis target_val_idx - N_WEEKS gehen.
            max_idx = len(X) + target_val_idx - N_WEEKS + 1
            if max_idx <= 0:
                continue
            X, y, persist = X.iloc[:max_idx], y[:max_idx], persist[:max_idx]
        elif mode == "all":
            pass # Behalte alles
            
        X_parts.append(X)
        y_parts.append(y)
        p_parts.append(persist)
        r_parts.append(np.full(len(persist), region, dtype=object))

    if not X_parts:
        return pd.DataFrame(), np.array([]), np.array([]), np.array([])
        
    X = pd.concat(X_parts, ignore_index=True)
    return X, np.vstack(y_parts), np.concatenate(p_parts), np.concatenate(r_parts)


## 3 · Train/Validierungs-Split

## 4 · Baselines (MAE wie Kaggle)

## 5 · LightGBM — 5 Wochen-Modelle

Ein Modell je Vorhersagewoche. Bewusst **konservative** Parameter (flache
Baeume, grosse Blaetter, L1/L2, Subsampling) gegen Overfitting auf den
~1,7 Mio. Fenstern. Early Stopping gegen den Region-Holdout.

In [ ]:
LGB_PARAMS = dict(
    objective="mae",
    n_estimators=1000,
    learning_rate=0.03,
    num_leaves=11,
    max_depth=5,
    min_child_samples=300,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.6,
    reg_alpha=3.0,
    reg_lambda=5.0,
    verbose=-1,
)

FOLDS = [0, 5, 10]
fold_maes = []
best_iters_per_week = {w: [] for w in range(N_WEEKS)}

for fold, offset in enumerate(FOLDS):
    print(f"\n{'='*40}\nFold {fold+1} (Offset {offset} Wochen)\n{'='*40}")
    
    X_tr, y_tr, _, _ = build_samples(train_df, mode="train", val_offset_weeks=offset)
    X_va, y_va, _, _ = build_samples(train_df, mode="val", val_offset_weeks=offset)
    
    if len(X_va) == 0:
        print("Nicht genug Daten für diesen Fold!")
        continue
        
    print(f"Train-Fenster: {len(X_tr):,} | Val-Fenster: {len(X_va):,}")
    
    val_preds = np.zeros_like(y_va, dtype=np.float64)
    for w in range(N_WEEKS):
        m = lgb.LGBMRegressor(**LGB_PARAMS, random_state=RANDOM_STATE + w + fold*100, n_jobs=-1)
        m.fit(
            X_tr, week_target(y_tr, w),
            eval_set=[(X_va, week_target(y_va, w))],
            eval_metric="mae",
            callbacks=[lgb.early_stopping(ES_ROUNDS, verbose=False)],
        )
        val_preds[:, w] = clip_scores(m.predict(X_va))
        best_iters_per_week[w].append(m.best_iteration_)
        
    lgb_mae = mae(y_va, val_preds)
    fold_maes.append(lgb_mae)
    
    print(f"LightGBM MAE: {lgb_mae:.4f}")

print(f"\n{'='*40}\nDURCHSCHNITT ÜBER ALLE FOLDS\n{'='*40}")
print(f"LightGBM MAE (nur Wetterdaten): {np.mean(fold_maes):.4f}")


## 6 · Finales Training & Submission

Neu-Training auf **allen** Regionen mit der je Woche validierten Baum-Anzahl
(`best_iteration`). Vorhersage je Region aus dem letzten Tag des
Testfensters, anschliessend Persistence-Blend.

In [ ]:
# Neu-Training auf ALLEN Regionen
X_all, y_all, _, _ = build_samples(train_df, mode="all")

final_models = []
for w in range(N_WEEKS):
    avg_best_iter = int(np.mean(best_iters_per_week[w])) if best_iters_per_week[w] else LGB_PARAMS["n_estimators"]
    
    m = lgb.LGBMRegressor(
        **{**LGB_PARAMS, "n_estimators": avg_best_iter},
        random_state=RANDOM_STATE + w,
        n_jobs=-1,
    )
    m.fit(X_all, week_target(y_all, w))
    final_models.append(m)
print(f"Finales Training: {N_WEEKS} Modelle auf {len(X_all):,} Fenstern")

# Eine Feature-Zeile je Region: letzter Tag des 91-Tage-Testfensters.
last_rows = test_df.sort_values(["region_id", "ordinal"]).groupby("region_id").tail(1)
test_regions = last_rows["region_id"].to_numpy()
X_test = last_rows[FEATURES]

test_preds = np.column_stack([clip_scores(m.predict(X_test)) for m in final_models])

# Keine Persistence mehr, reines Modell!
print("Submission: 100% LightGBM (Wetterdaten-Modell)")

submission = pd.DataFrame({"region_id": test_regions})
for k in range(N_WEEKS):
    submission[f"pred_week{k + 1}"] = test_preds[:, k]
submission = submission.sort_values("region_id").reset_index(drop=True)

out_path = SUBMISSION_DIR / f"submission_{MODE}_v2.csv"
submission.to_csv(out_path, index=False)
print(f"Gespeichert: {out_path}  ({len(submission):,} Zeilen)")
submission.head(10)
